In [2]:
from ultralytics import YOLO
import torch
import numpy as np
import matplotlib.pyplot as plt

In [3]:
#Load in YOLO model
#model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
model  = YOLO("yolo26n.pt")
model.info()
device = "cuda:0" if torch.cuda.is_available() else "cpu"
load_model = model.to(device)

YOLO26n summary: 260 layers, 2,572,280 parameters, 0 gradients, 6.1 GFLOPs


We want to iteratively go through each layer in the model and quantise its weights and activations

Weight quantisation uses a simple quantisation algorithm, can be: 
    - Linear
    - Logarithmic

Activation quantisation is more involved, steps are:
    - With original model, pass through calibration data and record activations throughout the model
    - Use the recorded activations and for each layer, find the dimension with the most outliers
    - With the dimension with the most outliers, group them using k-means and calculate there quantisation scale and zero point
    - With these values, for the respective layer, we can quantise the activations by which group they belong to.

In [4]:
import os

calibration = os.listdir("cocosample")


#Create map to save layer activations
activations = {}

#Create hook to save activations
def get_activations(name):
    def hook(model, input, output):
        activations[name] = output
    return hook

handles = []

#Register hooks to layers
for name, layer in load_model.named_modules():
    #print(name, layer)
    if len(list(layer.children())) == 0:
        handles.append(layer.register_forward_hook(get_activations(name)))

In [5]:
#Run inference on all sample images in cocosample
path = "C:/Users/anuhg/Documents/Southampton/_Lessons/Year4/DPDL/Group_Project/DGQ/cocosample/"
for img in calibration:#tqdm(range(len(calibration))):
    with torch.no_grad():
        model(path+img)#calibration[img])

for i in handles:
    i.remove()


image 1/1 C:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DGQ\cocosample\000000002473.jpg: 448x640 4 persons, 1 skis, 163.2ms
Speed: 21.5ms preprocess, 163.2ms inference, 4.6ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 C:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DGQ\cocosample\000000051610.jpg: 448x640 1 person, 2 beds, 1 laptop, 129.1ms
Speed: 2.2ms preprocess, 129.1ms inference, 0.2ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 C:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DGQ\cocosample\000000053624.jpg: 448x640 3 persons, 1 elephant, 133.7ms
Speed: 3.5ms preprocess, 133.7ms inference, 0.2ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 C:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DGQ\cocosample\000000081738.jpg: 480x640 1 person, 1 cake, 1 dining table, 153.4ms
Speed: 5.1ms preprocess, 153.4ms inference, 0.2ms postprocess per image at sh

In [6]:
#Define function for maximum activation dimension
def MaxActivationDimensionCNN(layer):
        #Layer will be an array of weights

        #Pixel Dimension
        pixel_dim = layer.view(1,-1).squeeze()
        pixel_var = max(pixel_dim) - min(pixel_dim)
        
        #Channel Dim
        channel_dim = layer.view((layer.shape[0], -1)) #channel is dependent on the output channels
        channel_var = (max(torch.max(channel_dim, 0)[0]) - min(torch.max(channel_dim, 0)[0])) + (max(torch.min(channel_dim, 0)[0]) - min(torch.min(channel_dim, 0)[0]))
        
        if channel_var > pixel_var:
            return 0, channel_dim
        else:
            return 1, pixel_dim
    
#Visualise on 2 layers
count = 0
for name, activation in activations.items():
    print(name, activation.shape)
    print(MaxActivationDimensionCNN(activation))
    if count >= 10:
         break
    count+=1

model.model.0.conv torch.Size([1, 16, 240, 320])
(0, tensor([[31.4447,  1.2760,  0.6469,  ...,  1.1282,  0.9147,  2.7637]]))
model.model.0.act torch.Size([1, 256, 15, 20])
(0, tensor([[-0.1947, -0.1834, -0.2350,  ...,  1.2734, -0.2134, -0.2781]]))
model.model.1.conv torch.Size([1, 32, 120, 160])
(0, tensor([[11.0427, -0.1004, -0.1316,  ...,  4.0261,  4.1843,  7.7662]]))
model.model.2.cv1.conv torch.Size([1, 32, 120, 160])


KeyboardInterrupt: 

In [ ]:
from sklearn.cluster import KMeans
import copy
from tqdm import tqdm
from torch import nn

class Quantizer:

    def __init__(self):

        self.statistics = {}

    def MaxActivationDimensionCNN(self, layer):
        #Layer will be an array of weights

        #Pixel Dimension
        pixel_dim = layer.view(1,-1).squeeze()
        pixel_var = max(pixel_dim) - min(pixel_dim)
        
        #Channel Dim
        channel_dim = layer.view((layer.shape[0], -1)) #channel is dependent on the output channels
        channel_var = (max(torch.max(channel_dim, 0)[0]) - min(torch.max(channel_dim, 0)[0])) + (max(torch.min(channel_dim, 0)[0]) - min(torch.min(channel_dim, 0)[0]))
        
        if channel_var > pixel_var:
            return 0, channel_dim
        else:
            return 1, pixel_dim

    def logarithmic_quantization(self, activations, q_s, q_z, num_bits=2):
        q_activations = torch.clamp(torch.round(-torch.log2(activations/q_s)), 0, pow(2, num_bits)-1)
        return (q_activations - q_z) *q_s

    def linear_quantization(self, activations, q_s, q_z, num_bits=2):
        q_activations = torch.clamp(torch.round(activations/q_s) + q_z , 0, pow(2, num_bits)-1)
        return q_s*pow(2,-1*q_activations)        
    
    def quantize(self, activations, quant_func, n_groups=2, n_bits=2): # Can speed up later by passing in layer reference too quantise weights live
        
        if len(activations.size()) < 2:
            acts = activations.unsqueeze(0)
        else:
            acts = activations.squeeze()

        q_activations = torch.zeros_like(acts)

        for i in range(len(acts)):
            channel = acts[i]
            #print(channel.shape)

            d_max = max(channel.view(1,-1))
            d_min = min(channel.view(1,-1))

            q_s = (d_max - d_min)/(pow(2,n_bits))
            z   = -d_min/q_s

            q_activations[i] = quant_func(channel, q_s, z, n_bits)       

        return q_activations

    def group_quantize(self, activations, quant_func, n_groups=2, n_bits=2): # Can speed up later by passing in layer reference too quantise weights live
        
        if len(activations.size()) < 2:
            acts = activations.unsqueeze(1)

        else:
            acts = activations

        q_activations = torch.zeros_like(acts)

        #Group channels
        kmeans = KMeans(n_clusters=n_groups, random_state=0, n_init="auto")
        kmeans.fit(acts)
        groups = kmeans.predict(acts)
    
        quant_groups = {}

        for i in range(len(groups)):

            if groups[i] in quant_groups: #if cluster centre already mapped

                if max(acts[i])>quant_groups[groups[i]]["max"]:
                    quant_groups[groups[i]]["max"] = max(acts[i])


                elif min(acts[i])<quant_groups[groups[i]]["min"]:
                    quant_groups[groups[i]]["min"] = min(acts[i])
            
            
            else: #if cluster centre hasnt been mapped yet
                quant_groups[groups[i]] = {"min": min(acts[i]), "max": max(acts[i])}


        quant_params = {}

        for group, params in quant_groups.items():
            q_s = (params["max"] - params["min"])/(pow(2,n_bits))
            z   = -params["min"]/q_s
            quant_params[group] = {"q_s":q_s, "z":z}

        # print(quant_params)

        #apply scaling
        for i in range(len(groups)):
            q_activations[i] = quant_func(activations[i], quant_params[groups[i]]["q_s"], quant_params[groups[i]]["z"], n_bits)       

        return q_activations
    
    def quantize_model(self, model, quant_method, quantizer, num_groups, num_bits):
        q_model = copy.deepcopy(model)

        progress_bar = tqdm(total=len(list(q_model.named_parameters())))

        for name, param in q_model.named_parameters():
            d, dim = self.MaxActivationDimensionCNN(param)
            param.data  = nn.parameter.Parameter(quant_method(dim.detach(), quantizer, num_groups, num_bits).view(param.shape))

            self.statistics[name] = "pixel_dim" if d else "channel_dim"
            progress_bar.update(1)

        return q_model

